# Basic Sec-Gemini SDK Usage

This notebook demonstrates the core operations of the Sec-Gemini Python SDK:
creating sessions, sending prompts, streaming responses, uploading files, and managing sessions.

## Setup

Install the SDK.

In [ ]:
!pip install -q sec-gemini

## Authentication

Set your API key. Get one from [secgemini.google/keys](https://secgemini.google/keys).

In [ ]:
import os

# Option 1: From environment variable
API_KEY = os.environ.get("SEC_GEMINI_API_KEY")

# Option 2: From Colab secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        API_KEY = userdata.get("SEC_GEMINI_API_KEY")
    except (ImportError, Exception):
        pass

# Option 3: Paste directly (not recommended for shared notebooks)
if not API_KEY:
    API_KEY = "YOUR_API_KEY_HERE"

assert API_KEY and API_KEY != "YOUR_API_KEY_HERE", "Please set your API key"

## Connect to Sec-Gemini

Create and start the client.

In [ ]:
from sec_gemini import SecGemini

client = SecGemini(api_key=API_KEY)
await client.start()
print("Connected to Sec-Gemini")

## Create a Session

Sessions are persistent conversations. They survive client disconnections.

In [ ]:
session = await client.sessions.create()
print(f"Session created: {session.id}")
print(f"Status: {session.status}")

## Send a Prompt and Stream Responses

Send a security question and watch the agent work in real time.

In [ ]:
await session.prompt("Check the email security posture of example.com")

async for msg in session.messages.stream():
    msg_type = msg.get("message_type", "")
    content = msg.get("content", "")
    title = msg.get("title", "")

    if msg_type == "MESSAGE_TYPE_RESPONSE":
        print(f"\n--- Agent Response ---\n{content}")
    elif msg_type == "MESSAGE_TYPE_THOUGHT":
        print(f"  [Thinking] {content[:150]}...")
    elif msg_type == "MESSAGE_TYPE_TOOL_CALL":
        print(f"  [Tool] {title}")
    elif msg_type == "MESSAGE_TYPE_TOOL_RESULT":
        print(f"  [Result] {content[:100]}...")
    elif msg_type == "MESSAGE_TYPE_AGENT_IS_DONE":
        print("\n--- Session Complete ---")

## List Sessions

You can list all your sessions to see their status.

In [ ]:
sessions = await client.sessions.list()
for s in sessions:
    print(f"  {s.id[:12]}...  {s.name:<30}  {s.status}")

## Upload a File

Upload a file to a session so the agent can reference it.

In [ ]:
import tempfile

# Create a sample file
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write(
        "Server: Apache/2.4.41\nX-Powered-By: PHP/7.4\nSet-Cookie: session=abc123; HttpOnly\n"
    )
    tmp_path = f.name

# Create a new session for the file demo
file_session = await client.sessions.create()

# Upload the file
await file_session.files.upload(tmp_path)
print(f"Uploaded {tmp_path}")

# List files
files = await file_session.files.list()
for fi in files:
    print(f"  File: {fi.filename}")

In [ ]:
# Ask the agent about the uploaded file
await file_session.prompt(
    "Analyze the HTTP headers in the uploaded file. Are there any security concerns?"
)

async for msg in file_session.messages.stream():
    msg_type = msg.get("message_type", "")
    content = msg.get("content", "")

    if msg_type == "MESSAGE_TYPE_RESPONSE":
        print(content)
    elif msg_type == "MESSAGE_TYPE_AGENT_IS_DONE":
        break

## Cleanup

Delete the sessions and close the client.

In [ ]:
await session.delete()
await file_session.delete()
await client.close()
print("Done!")